In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

from sklearn.model_selection import train_test_split

from TorchSpatial.trainer import train, train_sri_debias
from TorchSpatial.tester import test
from TorchSpatial.modules.encoder_selector import get_loc_encoder
from TorchSpatial.modules.models import ThreeLayerMLP
import TorchSpatial.utils.datasets as data_import
import TorchSpatial.utils.eval_helper as eval_helper

from gbsloss import SSIPartitioner, BinaryPerformanceTransformer, SSILoss, SRIPartitioner, SoftHistogramPerformanceTransformer, SRILoss

from pathlib import Path
import numpy as np
import pandas as pd

import torch
import numpy as np

import json

import warnings


In [3]:
all_data = data_import.load_dataset(params = {"dataset": "birdsnap", "meta_type": "ebird_meta", "regress_dataset": []},
        eval_split = "test",
        train_remove_invalid = True,
        eval_remove_invalid = False,
        load_cnn_predictions=True,
        load_cnn_features=False,
        load_cnn_features_train=False)


Loading birdsnap_with_loc_2019.json - train
   using meta data: ebird_meta
	 46386 total entries
	 43426 entries with images
	 42490 entries with meta data
Loading birdsnap_with_loc_2019.json - test
   using meta data: ebird_meta
	 2443 total entries
	 2262 entries with images
	 2217 entries with meta data
	 keeping entries even without metadata


In [4]:
all_data.keys()

dict_keys(['train_locs', 'val_locs', 'train_preds', 'train_classes', 'train_users', 'train_dates', 'train_inds', 'train_imgs', 'val_classes', 'val_users', 'val_dates', 'val_inds', 'val_imgs', 'class_of_interest', 'classes', 'num_classes', 'val_preds', 'val_split'])

In [5]:
all_data["train_preds"].shape # This is the cnn image prediction for training

(43426, 500)

In [6]:
all_data["val_preds"].shape # This is the cnn image prediction for validation

(2262, 500)

In [11]:
all_data["val_preds"][170]

array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e+00,
       0.0000000e+00, 1.7273960e-06, 4.1142102e-05, 2.7263520e-06,
       5.0794383e-06, 1.5173841e-04, 0.0000000e+00, 0.0000000e+00,
       1.8991213e-04, 7.1914430e-05, 1.1283653e-06, 4.3496152e-06,
       1.9363214e-04, 2.2858849e-02, 2.4051279e-04, 1.4847105e-03,
       9.7452521e-01, 3.3960200e-06, 2.7185619e-05, 2.0818118e-06,
       1.0291435e-06, 2.9843966e-06, 2.2626321e-06, 1.0603761e-06,
       1.3172955e-06, 1.0495486e-06, 0.0000000e+00, 5.7713019e-06,
       4.3320383e-06, 3.4168211e-06, 1.6155674e-06, 0.0000000e+00,
       0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 0.0000000e

In [10]:
all_data["val_preds"][170].argmax()

np.int64(40)

In [13]:
all_data["val_preds"][170][all_data["val_preds"][170].argmax()] # max proba

np.float32(0.9745252)

In [15]:
all_data["val_preds"][3].max() # max proba

np.float32(0.6652693)

In [16]:
all_data["val_preds"][170].argmax() == all_data["val_classes"][170] # compare cnn preds to correct class, find accuracy

np.True_

In [75]:
import random
import numpy as np
truths = []

# Accuracy of 5000 randomly sampled pairs with replacement
for _ in range(5000):
    i = random.randint(0, len(all_data["val_preds"]) - 1)
    truths.append(all_data["val_preds"][i].argmax() == all_data["val_classes"][i])
truths = np.array(truths)
print(f"accuracy: {len(truths[truths == True]) /  len(truths)}")

accuracy: 0.6998


In [ ]:
print(f"accuracy: {len(truths[truths == True]) /  len(truths)}")

[70s Choir Vibing Rhythmic Music]

La, So La Do La!

Hey, You computer!

La la la So La Do La!

It's time to print the numbers!

La la So La Do La,

Calculate the fraction,

La, 

Hey,

Re Do Mi, Re Do Mi!

Listen up, listen up!

[You're still rolling stones chorus]

Let truths be told, compared around;

Filter the Frame, let len it bound;

Dividing by the length of truths and see

Accuracy be stored. 


In [ ]:
truths = np.array(truths)

One became two, as one walked through the mirror and two copies emerged from each side. 

In [ ]:
pred = class_probas.argmax(dim=1)
hit_at_1 = (pred == y_idx)

I reported seeing some confusion among the audience, who demanded a change of names so they may better recognize the variables. I completel agreed with them, so I likewise demanded the change, and subsequently altered the old context to fit the new faces. They looked better immediately, and I was impressed of myself for having pulled this off all live on stage. 

In [97]:
class_probas = torch.Tensor([all_data["val_preds"][5], all_data["val_preds"][6], all_data["val_preds"][7]])
print(class_probas.shape)
pred = class_probas.argmax(dim=1)
y_idx = torch.Tensor([all_data["val_classes"][5], all_data["val_classes"][6], all_data["val_classes"][7]])
hit_at_1 = (pred == y_idx)
print(hit_at_1)

torch.Size([3, 500])
tensor([False, False, False])


In [182]:
top1accuracies = []

for _ in range(100):
    class_probas = []
    y_idx = []

    for _ in range(200):
        i = random.randint(0, len(all_data["val_preds"]) - 1)
        class_probas.append(all_data["val_preds"][i])
        y_idx.append(all_data["val_classes"][i])

    pred = torch.tensor(class_probas).argmax(dim=1)
    y_idx_tensor = torch.tensor(y_idx)
    hit_at_1 = (pred == y_idx_tensor)

    # print(hit_at_1[:6])
    top1accuracy = hit_at_1.float().mean().item()
    top1accuracies.append(top1accuracy)

    # print(f"Top 1 Accuracy: hit_at_1 {top1accuracy:.4f}")
    # print((hit_at_1 == True).sum().item() + (hit_at_1 == False).sum().item() == len(hit_at_1))

print(top1accuracies[:6])
print(np.mean(top1accuracies))

[0.7300000190734863, 0.7049999833106995, 0.699999988079071, 0.7200000286102295, 0.6899999976158142, 0.6800000071525574]
0.6961000025272369


Seeing my code being glowed up by AI, I feel both relieved and confused. It was not in my own style. It is different. I must read it again. 

In [ ]:
# print(f"Top 1 Accuracy: hit_at_1 {top1accuracy:.4f}")

The computer receives a request to print the message according to the provided declaration, only after it formatted the form with the lastest information it received from the variable top1accuracy, and trimmed away all digits after the fourth after the decimal dot. 

In [ ]:
top1accuracies = []

In front of the vast and empty space of the background of a comfortably colored darkness, I carefully, meticulously, precisely arranged a decent room amidst the orderly chaos in the random access memory in preparation for the soon-expected growth in registered membership count belonging to the name that I have just previously decided and set. I prepared it in response to the prophecy on the birth of many who will be of immense value to the mission I have accepted and undertook, whom I must seek and gather immediately after their arrival, lest I return an iteration later and fail to find them, only to realize their unfortunate but predetermined erasure by next-generation inhabitants arriving carrying the exact same names.

In [ ]:
for _ in range(100):

I dispatched a trivial messager, a variable whose name might as well be nonexistent, so the counter for the loop may work. Like all good variables, it would firmly hold onto its assigned value upon each announcement, to carry its important flag of one specific integer to synchronize the knowledge of the state of iterations for all who would demand to receive an update on the most recent progress. 

(for i would be adding: "so it might hop through the various reserved positions hidden in the maze-like steps installed into the subsequent lines of the looping program")

In [ ]:
i = random.randint(0, len(all_data["val_preds"]) - 1)

In my right hand I hold a form that I wish I can already send out. But until things spin into motion, until I am granted live permission to measure the length of this specific division of the data, I will not have my upper bound, which is a necessary piece of information that will be requested from my submission by the Office of the Library of Artificial Randomness once they begin to process my "Random Integer Request". Until then, until the office would return its integral response, the identity of i would remain a bounded mystery, and I can only wait and press down my burning desire to unveil it. 

In [ ]:
class_probas.append(all_data["val_preds"][i])
y_idx.append(all_data["val_classes"][i])

I would command, "Come out, you the i-th member of the list belonging to the "val_preds" division of the all_data group!" and it would come out from its place. It is a list of mutually exclusive probabilities, as expected. I would append it to the end of the class_probas group, the collection I built for all whom I will sample from the same division of the same group. I would do similarly for the class labels, calling "you i-th member of the list belonging to the "val_classes" division of the all_data group" and append it to another group -- y_idx. Each would abide by my order and stay in its place. In fact, I would be quite terrified if any one of them moved on its own, for that would imply that another secret commander -- or disrupter -- or confounding factor, was also present. 

In [ ]:
preds = torch.tensor(class_probas).argmax(dim=1)

It is unfortunate that I must proceed with an immigration request, but a Python list is truly too primitive to support even an operation such as an argmax with a dimensional specification. Fortunately, in the past, as long as I had correctly sent over what it could process, torch's tensor conversion unit had always received my input, performed the argmax search, and returned a single array of single integers, each of which indicated the position of the largest probability in each list of probabilities which lie along dimension one, all flawlessly without fail. I would gladly receive its processed tensor, and nickname it "preds". 

In [ ]:
y_idx_tensor = torch.tensor(y_idx)

y_idx passed through the gate of conversion, and subsequently received a suffix indicating its new data structure, a suffix that hints toward a different version of itself that exists somewhere else. 

In [ ]:
hit_at_1 = (pred == y_idx_tensor)

The two tensors named pred and y_idx_tensor were aligned. Sharing the same shape allowed them to bijectively correspond their entries and thus compare element-wise. The boolean rules governing the comparison are simple: 

Let a match be True, a mismatch be False; 

Collect them all, and keep the shape; 

At each entry, its status lays; 

Name it hit_at_1, so it stays. 

In [ ]:
top1accuracy = hit_at_1.float().mean().item()

Before it is just a number, it was a tensor holding the number; before it held that single number, it held many numbers; before it held the many numbers, it held them as boolean markers. Through a sequence of transformational portals it became what it is, a Python float rightfully named top1accuracy, representing the percentage of correctness that is the aggregate of every individual True and False divided by the size of the selected batch, according to one of the strictest, merciless criterion of judgement that can be reasonably implemented, analogous to a multiple choice response that is restricted to a single option only: hit_at_1. 

In [ ]:
top1accuracies.append(top1accuracy)

The list mechanically received another number, oblivious to the programmer's intent in heart. 

In [ ]:
print(top1accuracies[:6])

In order to prevent revealing a whole screen of numbers, I restricted to display only the first six elements of the list. Not everything stored must be seen.

In [ ]:
print(np.mean(top1accuracies))

I averaged across the floats in the list called top1accuracies, invisibly making a numpy copy in the process. 

In [90]:
class_probas_based_on_image = torch.from_numpy(all_data["val_preds"][5])
class_probas_based_on_image[:6]

tensor([0.0010, 0.0010, 0.0058, 0.0132, 0.0043, 0.0030])

In [89]:
class_probas_based_on_loc = loc_embedding = torch.ones_like(class_probas_based_on_image).float()
class_probas_based_on_loc[:6]

tensor([1., 1., 1., 1., 1., 1.])

In [91]:
(class_probas_based_on_image * class_probas_based_on_loc)[:6]


tensor([0.0010, 0.0010, 0.0058, 0.0132, 0.0043, 0.0030])

In [93]:
all(class_probas_based_on_image * class_probas_based_on_loc == class_probas_based_on_image) # simulating no_prior. Should be true before and after, because loc_probas are all 1.

True

Check the radian of input data! tensor([-92,  46])

Training for 0 epochs.

Training Completed.

Model saved as TorchSpatial/pre_trained_models/no_prior/model_birdsnap_ebird_meta_no_prior_trained0_debiased0.pth.tar; in total, trained for 0 epochs, debiased for 0 epochs, in the order of []

Top-1 Accuracy on 2217.0 test images: 0.18%

Top-3 Accuracy on 2217.0 test images: 0.41%

MRR on 2217.0 test images: 0.0125

I witnessed the outputs, and my mind was full with conflicting assumptions, yet all of them in unison prompted me to ask this single question: What went wrong? 

- I prefer [lat, lon], but it has [lon = -92, lat = 46]. 
- Top-1 Accuracy and Top-3 Accuracy are both too low, and MRR is also too low, suggesting a fundamental mismatch between the cnn predictions and the correct classes. 

[I used to simply let my assumptions sway -- not a good intuition for project consistency.]

In this last stage that is evaluation, I have no embedding_loss, no [], no trainable parameter whatsoever. All I have is a testloader testing a static list against another static list; all I had to do was making sure they arrive at each location at the correct order in time and space. 

# The Journey of top1_acc

In [ ]:
img_te = torch.Tensor(all_data["val_preds"]).long() # shape=(2262, 500)
test_data_zip = list(zip(idx_te, img_te, loc_te, y_te))
test_loader  = DataLoader(test_data_zip, batch_size=batch_size, shuffle=False)
rows = test(test_loader,
                    loc_encoder,
                    # ssi_loss,
                    # test_ssi_partitioner,
                    # test_ssi_perf_transformer,
                    # sri_loss,
                    # test_sri_partitioner,
                    # test_sri_perf_transformer,
                    # scale_grid,
                    # distance_lag,
                    # split_number,
                    device)
def test(dataloader,
         loc_encoder,
        #  ssi_loss,
        #  ssi_partitioner,
        #  ssi_perf_transformer,
        #  sri_loss,
        #  sri_partitioner,
        #  sri_perf_transformer,
        #  scale_grid,
        #  distance_lag,
        #  split_number,
         device):
    for idx_b, img_b, loc_b, y_b in dataloader:
        img_b = img_b.to(device)
        class_probas_based_on_image = img_b
        class_probas = torch.mul(class_probas_based_on_loc, class_probas_based_on_image)
        pred = class_probas.argmax(dim=1)
        correct_top1 += (pred == y_b).sum().item()
        top1_acc = 100.0 * correct_top1 / total if total else 0.0
        print(f"Top-1 Accuracy on {total} test images: {top1_acc:.2f}%")



On the collage in front of me, I expected to see a shorter route than I would have seen elsewhere, and I did. Each scrap on this piece of artwork is a section of syntax-highlighted Python gathered from places along top1_acc's journey -- other files in the region. The variable didn't travel very far at all. In fact, in this particular case, I only needed to visit two production pipelines, one a literal neighbor, and the other one block down. It was a very short walk for me, and took less than five minutes, before I could even get a sense of just how the air here and there really differ in smell. Then, here I am, appraising this codeblock that is completely cut up and unoperable, and mining for insights of this past incident. What really happened? I want to know. If the artwork wouldn't tell me, I can always walk top1_acc's path myself; but what I have is sufficient for now.

In [ ]:
img_te = torch.Tensor(all_data["val_preds"]).long() # shape=(2262, 500)

I observed the observable origin of its journey: the gate of the dataset, from where the values were imported, packaged as a torch tensor, and turned into long floats, or int 64. 

It was at this point that I received my relevation. *Wait a second.* Did it just turn probabilities into int 64, either 0 or 1, when their nature of being probabilities were most crucial to computing the accuracy? *This MUST be fixed, and I hope the fix fixes all.*

[My wandering mind reminded me that previous attempts with .long() in place did not cause the image embeddings, which used to serve at this section of the pipeline, to be unusable. I point out, however, that the image embeddings are numbers much larger than 1, and neither floor nor ceiling would matter to each number.]

Check the radian of input data! tensor([-92.0315,  46.8470])
Training for 0 epochs.
Training Completed.
Model saved as TorchSpatial/pre_trained_models/no_prior/model_birdsnap_ebird_meta_no_prior_trained0_debiased0.pth.tar; in total, trained for 0 epochs, debiased for 0 epochs, in the order of []
Top-1 Accuracy on 2217.0 test images: 70.09%
Top-3 Accuracy on 2217.0 test images: 86.69%
MRR on 2217.0 test images: 0.7904

Oh, seeing the calming numbers of 70.09%, 86.69%, and 0.7904, how relieved am I! It is such a simple mistake that was unfortunately overlooked for days. Had I been just a little better at using the most fundamental transformations available in the libraries, I would not have had commited this blatantly unacceptable numerical atrocity. 

"Your issue is resolved." I messaged top1_acc.

"Already?" top1_acc replied in disbelief.

"Already." I assured him. 

I thought "Suddenly, I decided to take the collage, and take a walk.", but it did not happen.

# SSI and SRI Scores

Top-1 Accuracy on 2217.0 test images: 70.09%
Top-3 Accuracy on 2217.0 test images: 86.69%
MRR on 2217.0 test images: 0.7904
SSI score on 0 valid test neighborhoods: 0.0000
SRI SG score on 1986 valid test neighborhoods: 0.0000
SRI DL score on 1999 valid test neighborhoods: 0.0000
SRI DS score on 1999 valid test neighborhoods: 0.0000

Since I first joined this project, I had never ceased to wonder why the SSI scores in the new implementation was at least an order of magnitude lower than those which are displayed in TorchSpatial. Not 16, but 1.6, or even lower. 
- Were not the underlying calculations and the data the same? Why then were they so different? 
- Was there a structural change in the pipeline?

The 0.0000 entries further baffled me. Without a location encoder SSI scores should still be computable, based on the distribution of the locations, and the correctness of the prediction at each point. The binary weights resulting from comparing each estimated probability (ex. 0.75) to a threshold (ex. 0.7, in which case this is "high" prediction) are produceable. The score would simply measure the geo-bias of the CNN model that was pretrained and not included. Since they are theoretically possible, yet their values are missing, something must be wrong in the implementations, which require yet a collage appreciation session to formally identify. 

Introducing: The Journey of ssi

In [ ]:
ssis = []
tmp_ssi, ignore_ratio = None, None
tmp_ssi, ignore_ratio = ssi_loss(neighborhood_points, neighborhood_values)
if tmp_ssi is not None:
    tmp_ssi = float(tmp_ssi[0].item())
    ssis.append(tmp_ssi)
ssi = np.mean(ssis) if len(ssis) > 0 else 0.0
print(f"SSI score on {len(ssis)} valid test neighborhoods: {ssi:.4f}")

In [ ]:
if tmp_ssi is not None:
    ssis.append(tmp_ssi)
ssi = np.mean(ssis) if len(ssis) > 0 else 0.0

Since ssi:.4f == 0.0000,

then len(ssis) <= 0,

then ssis == [],

then tmp_ssi was always None,

which is problematic.

The code below leads to the None being returned.

In [ ]:
class SSILoss(nn.Module):
    def forward(self, points, values):
        cs, ns = np.unique(values.detach().cpu().numpy(), return_counts=True)
        rmax = np.argmax(ns)
        # Handling extreme cases
        ignore_ratio = ns[rmax] / np.sum(ns)
        # print(f"Ignore ratio: {ignore_ratio}")
        if ignore_ratio > 0.9 or ignore_ratio < 0.6:
            return None, ignore_ratio

In [ ]:
class SSIPartitioner():
    def __init__(self, coords, k=100, radius=0.01, min_dist=0.0):
        self.coords = coords
        self.neighbors = self._construct_tree()
        self.backgrounds = self._construct_backgrounds()
    def get_neighborhood_points(self, idx):
        return self.coords[self.neighbors[idx]]
ssi_partitioner = SSIPartitioner(np.array([lats, lons]).T, k=partition_k, radius=ssi_radius)
class BinaryPerformanceTransformer(nn.Module):
    def __init__(self, thres=0.7):
        super(BinaryPerformanceTransformer, self).__init__()
        self.thres = thres

    def forward(self, logits, y):
        probs = torch.softmax(logits, dim=1)
        steps = torch.clamp(probs - self.thres, min=0.) / (probs - self.thres)

        return steps[np.arange(logits.shape[0]),y]
ssi_perf_transformer = BinaryPerformanceTransformer(thres=BinaryPerformanceTransformer_thres)
neighborhood_points = ssi_partitioner.get_neighborhood_points(idx.item())
neighborhood_values = ssi_perf_transformer(logits = class_probas, y = y_n)
tmp_ssi, ignore_ratio = ssi_loss(points = neighborhood_points, values = neighborhood_values)
class SSILoss(nn.Module):
    def forward(self, points, values):
        cs, ns = np.unique(values.detach().cpu().numpy(), return_counts=True)

An addition of an argument keyword sparked my intuitive recognition to guess that logits = class_probas is the culprit that turned the values wrong. No, no, no, logits are to be used, it it says logits. Let's navigate to the origin of this statement and resolve the discrepancy by putting back what should be there. But how can it be done in the no_prior case? They come as probas, not logits. Therefore, I must change my solution. Perhaps I can modify something in ssi_perf_transformer == BinaryPerformanceTransformer such that is takes probas as well as logits? Let's proceed. 

In [ ]:
class BinaryPerformanceTransformer(nn.Module):
        def forward(self, logits, y):
                probs = torch.softmax(logits, dim=1)

Clearly, however, the current structure already supports probas as well as logits. Passing an array of probas through a softmax would not change anything after all.

In [ ]:
ignore_ratio = ns[rmax] / np.sum(ns)
        # print(f"Ignore ratio: {ignore_ratio}")
        if ignore_ratio > 0.9 or ignore_ratio < 0.6:
            return None, ignore_ratio

I shall attempt to disable ignore_ratio for now, and see whether the output improves. Within a minute I will see the answer. 

Top-1 Accuracy on 2217.0 test images: 70.09%
Top-3 Accuracy on 2217.0 test images: 86.69%
MRR on 2217.0 test images: 0.7904
SSI score on 964 valid test neighborhoods: 0.0000
SRI SG score on 1986 valid test neighborhoods: 0.0000
SRI DL score on 1999 valid test neighborhoods: 0.0000
SRI DS score on 1999 valid test neighborhoods: 0.0000

It did not help. Thus, let's debug again.

It seems that one collage is not enough anymore. The ssi_loss(neighborhood_points, neighborhood_values) branch needs its own origin story.  

In [ ]:
# I shall assume SSILoss is implemented correctly.
ssi_loss = SSILoss()
def test(
        # dataloader,
        #  loc_encoder,
        ssi_loss,
        #  ssi_partitioner,
        #  ssi_perf_transformer,
        #  sri_loss,
        #  sri_partitioner,
        #  sri_perf_transformer,
        #  scale_grid,
        #  distance_lag,
        #  split_number,
        #  device
        ):
tmp_ssi, ignore_ratio = ssi_loss(neighborhood_points, neighborhood_values)

In [ ]:
class SSILoss(nn.Module):
    def __init__(self):
        super(SSILoss, self).__init__()
        self.analytical_surprisal = AnalyticalSurprisal()
    def forward(self, points, values):
        """
        :param points: the locations of the center and neighbor points, Bx2.
        :param values: the discretized performance values, B.
        :return: SSI loss.
        """

        cs, ns = np.unique(values.detach().cpu().numpy(), return_counts=True)
        rmax = np.argmax(ns)

        ignores = np.ones_like(cs)
        ignores[rmax] = 0

        # Handling extreme cases
        ignore_ratio = ns[rmax] / np.sum(ns)
        # print(f"Ignore ratio: {ignore_ratio}")
        if ignore_ratio > 0.9 or ignore_ratio < 0.6:
            return None, ignore_ratio

        weight_matrix = construct_weight_matrix(points, 4)

        loc, scale = self.analytical_surprisal.fit(cs, ns, weight_matrix, ignores)

        w_map = torch.from_numpy(weight_matrix).to(device=values.device, dtype=values.dtype)

        X = values.flatten().reshape((1, -1)) - torch.mean(values)

        Y = torch.matmul(w_map, X.T)

        moran_I_upper = torch.matmul(X, Y).flatten()

        low = torch.min(moran_I_upper, 2 * loc - moran_I_upper)

        prob = (1 + torch.erf((low - loc) / (scale * math.sqrt(2))))

        return -torch.log(prob + 1e-32), ignore_ratio

Loading birdsnap_with_loc_2019.json - train
   using meta data: ebird_meta
         46386 total entries
         43426 entries with images
         42490 entries with meta data
Loading birdsnap_with_loc_2019.json - test
   using meta data: ebird_meta
         2443 total entries
         2262 entries with images
         2217 entries with meta data
Check the radian of input data! tensor([-92.0315,  46.8470])
Training for 0 epochs.
Training Completed.
Debiasing for 0 epochs.
Debiasing Completed.
Model saved as TorchSpatial/pre_trained_models/space2vec-grid/model_birdsnap_ebird_meta_Space2Vec-grid_trained0_debiased0.pth.tar; in total, trained for 0 epochs, debiased for 0 epochs, in the order of []
Top-1 Accuracy on 2217.0 test images: 70.37%
Top-3 Accuracy on 2217.0 test images: 86.60%
MRR on 2217.0 test images: 0.7918
SSI score on 0 valid test neighborhoods: 0.0000
SRI SG score on 1986 valid test neighborhoods: 0.0000
SRI DL score on 1999 valid test neighborhoods: -0.0000
SRI DS score on 1999 valid test neighborhoods: -0.0000

What happened to the space2vec-grid model?

Model saved as TorchSpatial/pre_trained_models/space2vec-grid/model_birdsnap_ebird_meta_Space2Vec-grid_trained3_debiased1.pth.tar; in total, trained for 3 epochs, debiased for 1 epochs, in the order of [('train', 3), ('debias', 1)]
Top-1 Accuracy on 2217.0 test images: 69.78%
Top-3 Accuracy on 2217.0 test images: 86.20%
MRR on 2217.0 test images: 0.7879
SSI score on 0 valid test neighborhoods: 0.0000
SRI SG score on 1986 valid test neighborhoods: -0.0000
SRI DL score on 1999 valid test neighborhoods: -0.0000
SRI DS score on 1999 valid test neighborhoods: -0.0000

Apparently, training for 3 epochs and debiasing for 1 epoch has worse performance than random initialization. 

iased for 3 epochs, in the order of [('train', 30), ('debias', 3)]
Top-1 Accuracy on 2217.0 test images: 70.41%
Top-3 Accuracy on 2217.0 test images: 86.47%
MRR on 2217.0 test images: 0.7916
SSI score on 0 valid test neighborhoods: 0.0000
SRI SG score on 1986 valid test neighborhoods: 0.0000
SRI DL score on 1999 valid test neighborhoods: -0.0000
SRI DS score on 1999 valid test neighborhoods: 0.0000

iased for 3 epochs, in the order of [('train', 30), ('debias', 3)]
Top-1 Accuracy on 2217.0 test images: 70.41%
Top-3 Accuracy on 2217.0 test images: 86.47%
MRR on 2217.0 test images: 0.7916
SSI score on 0 valid test neighborhoods: 0.0000
SRI SG score on 1986 valid test neighborhoods: 0.0000
SRI DL score on 1999 valid test neighborhoods: -0.0000
SRI DS score on 1999 valid test neighborhoods: 0.0000

Ensure [lat, lon] consistency!!!

Make training more effective with decaying learn rates. 

SSILoss points: [[ 0.7395993  -1.2476147 ]
 [ 0.7395993  -1.2476147 ]
 [ 0.74090165 -1.2461897 ]
 [ 0.7381975  -1.2499306 ]
 [ 0.7381975  -1.2499306 ]
 [ 0.73759985 -1.2495545 ]
 [ 0.74133104 -1.2449348 ]
 [ 0.7393038  -1.2435223 ]
 [ 0.7395134  -1.241732  ]
 [ 0.741208   -1.2419456 ]
 [ 0.74140364 -1.2418419 ]
 [ 0.7387649  -1.240974  ]
 [ 0.73640084 -1.2400303 ]
 [ 0.7452032  -1.254822  ]
 [ 0.74392754 -1.2380251 ]
 [ 0.73217314 -1.2529765 ]
 [ 0.7416431  -1.2363945 ]
 [ 0.74354464 -1.2360259 ]
 [ 0.7463159  -1.237702  ]]
SSILoss values: tensor([-0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.])
ignore_ratio > 0.9 or ignore_ratio < 0.6

Clearly, the SSILoss values: tensor([-0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.]) is wrong.

I see it. I inputted probas now instead of logits, and that broke it. 

Apply Softmax to probabilities will turn probabilities small. 

So I branched within the PerfTransformers, so it only applies softmax to logits.

What took me a few hours to pinpoint took Chat a few seconds -- once I gave it the context and the exact files to look for. 

#######

# Loading Checkpoints

RuntimeError: Error(s) in loading state_dict for LocationEncoder:

<span style="color: pink;">

- size mismatch for spa_enc.ffn.layers.1.linear.weight: copying a param with shape torch.Size([256, 512]) from checkpoint, the shape in current model is torch.Size([**500**, 512]).
- size mismatch for spa_enc.ffn.layers.1.linear.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([**500**]).
- size mismatch for class_emb.weight: copying a param with shape torch.Size([500, 256]) from checkpoint, the shape in current model is torch.Size([500, **500**]).
- size mismatch for user_emb.weight: copying a param with shape torch.Size([5763, 256]) from checkpoint, the shape in current model is torch.Size([1, 500]).

</span>


In [ ]:
"spa_embed_dim": 500,

The last moments are coming. I renamed an old checkpoint that I had closely examined around a month ago, and loaded it with my current script. The matching of words took merely the assistance of a text editor and two of my own eyes, and I simply altered a few letters in my configs.json, and the entrance was cleared. The checkpoint was welcome in smoothly with no error in the unpacking process. Miracuously, the workers found a corresponding piece for each part within the checkpoint, be it a wing, a wheel, a control room, a commercial seating area, or something else. Only the sizes differ. I gave a cheerful shout that was echoed by my team throughout the garage. It worked; all the fixing and patching and observing the original structures worked. After almost sixty days of intense stuggle and risk taking and running marathons under deadline pressure, we had finally collaborative put together a prototype that matched the original in structure. Among us were Chat and Gemini, clapping. 

What's left to do aren't difficult either. A brief glance at the terminal output convinced me that the first three issues are trivially solvable by altering a single number in the setup. The fourth, however, unsettles me, for it is like a blackhole on a programming schedule, and I cannot estimate how long it may take me to solve. Maybe 5 minutes, maybe 5 hours. It involves checking the embedding loss equation once again and ensure that my setups are neither more nor less rigid than the math required. 

Currently I suspect an undue rigidity hidden in my code about the object embedding. While a class embedding for Birdsnap ought to have 500 classes (columns), it should be able to have an arbitrary size of embedding size (rows). However, I was constrained to 500 x 500 for birdsnap, no more, no less. Some matrix multiplications may be mismatched, causing a matrix to be erroneously transposed (or lacking being transposed). 

Also, although not used, it would best for the future if I also properly treat user_emb.weight, even if I would not need it in this experiment. 

RuntimeError: Error(s) in loading state_dict for LocationEncoder:
- size mismatch for class_emb.weight: copying a param with shape torch.Size([500, 256]) from checkpoint, the shape in current model is torch.Size([500, 500]).
- size mismatch for user_emb.weight: copying a param with shape torch.Size([5763, 256]) from checkpoint, the shape in current model is torch.Size([1, 500]).

It was just a quick fix, and it was meant to stay that way. I decided confidently because I knew its incompetency wouldn't remain hidden forever. There is a task in my future at where I will surely arrive, and once I flip the sequence of switches correctly, the automation built into this programming language will respond with precise enough a hint that point to exactly the place I will need to be. 

After altering a single number in main_ssi.py, there is only one error left.

RuntimeError: Error(s) in loading state_dict for LocationEncoder:
- size mismatch for user_emb.weight: copying a param with shape torch.Size([5763, 256]) from checkpoint, the shape in current model is torch.Size([1, 256]).

Simply load the users data, init the user embeddings with len(users), and fill the user embeddings with provided weights. 

In [ ]:
all_data = data_import.load_dataset(params = {"dataset": "birdsnap", "meta_type": "ebird_meta", "regress_dataset": [], "rand_sample_weight": 0.3},
    eval_split = 'test',
    train_remove_invalid = True,
    eval_remove_invalid = True,
    load_cnn_predictions=True,
    load_cnn_features=True,
    load_cnn_features_train=True)
len(set(all_data["train_users"])) # 5763

Now I found the number through questions like tall grasses which cannot stop anyone who is good at mathematics (problems like finding uniques using set()). I held it in front of me my right hand like how Martha (from The Secret Garden) would have held a plate of candle through the hallway at night, or a cave explorer in a Mayan maze would have held a metal detector (ref. Tintin). Everywhere I saw were ports (ref. international space station), some old ethernet cables, some for USB-C, some for screwdrivers, and some for [some other strange kind of non-digital port]. There are receivers of other objects, some a single number, a single variable, while others highly specific, some even more than TorchSpatial's dataset reader. I walked in the concrete tunnels like I explored natural caves. Everywhere there was an abundance of objects, only the drops of water and the flapping bat wings were replaced by sounds of buzzers and computer readers, and lights of my glowing number were against not a backdrop of darkness and watery reflection but indicator lights of red and green like stars in the sky. All the information here could not be traversed in a decade just by myself. I needed to find a helpful assistant to guide me to where it is. 

I observed the map on my hand, and traced to the set up of a (model.py) LocationEncoder, whose cloaky wrap of the instance variables contains what I need to see. Soon, I stood inside the elevated projection, and observed the class's functionalities, which previously felt overwhelming, but now felt almost too few and too sparse when arranged in 3D space. I looked above me, and there it was: the self.user_emb, whose definition hung there on its side, highlighted to be as clear as day. I would only need to known where to put my number, after which I will join the pipeline, and finish it all. 

Yes! [a view of celebrating at the blue sky with large chunks of clouds]

# After Successfully Loading Checkpoints

Apparently, 

An old checkpoint, after being ran just once, will be replaced by a new version of itself, if no further training is done. I instead want nothing to be changed if no training ever happened. 

Check: over two subsequent, identical runs, >>> old version <<< persist -> It does!



Now there is a somewhat boring task, that I should rename all previous checkpoints manually, and verify that they can all be imported, trained, debiased, and stored. That would surely be enough to take an entire afternoon.

In [ ]:
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x500 and 256x500)

But plans are always plans. Just as I was saying it, the birdsnap ebird_meta space2vec-grid pipeline crashed immediately. Behold, it was an old issue, which resurfaced in the way as I had predicted. A matrix multiplication, involving class embeddings and loc_embeddings, based on equation 7 in Aodha's 2019 paper, could not proceed, because there was a dimensional mismatch, an unfortunate difference, which is further obscured by the amount of possibilities generated by the several variables and their semantics. 

So, what are the two matrices? And which is mat1, which is mat2?

In [ ]:
File "/Users/bolongtang/GeoBS_push/GeoBS/baselines/TorchSpatial/utils/losses.py", line 445, in embedding_loss
    loc_pred = torch.sigmoid(model.class_emb(loc_emb))

Let's print them. It's like being a traffic police standing at a specific section of the road to stop a specific car to inspect. 

In [ ]:
class_emb: out_dim: 500, in_dim: 256, or 500 by 256 matrix
loc_emb: torch.Size([32, 500])
RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x500 and 256x500)

Apparently, it seems that loc_emb is in the fault here for being strict about its size being the class count itself, here 500. Rather, it should have a size equal to the embedding dimension of class_emb, here 256. I shall head to main_ssi.py to debug. 

It's something within the model that was wrong. 
It [the LocationEncoder model] primitively fetched from the num_classes (500) from the params I provided, which should never have happened. I need to change its behavior but changing what I hand it. But what I fed it seemed correct, therefore it may be the LocationEncoder itself that was problematic. If I change it, it would no longer be backward compatible ... 

But all it took was putting back something I had set aside. # , return_feats=True
Now it is back. 
loc_emb_cat = model(loc_cat, return_feats=True)
No matrix multiplication issue occurred ever again. 
Two hours ago, I was in the mire of uncertainty between the confidence interval covering from 5 minutes and 5 hours. Now I knew I landed right in the middle. 